In [1]:
println("$(Threads.nthreads()) Threads")
if Threads.nthreads() == 1
    println("Not running in parallel")
end

8 Threads


In [2]:
# import Pkg; Pkg.add("JLD2")

In [3]:
using Plots
using StatsPlots
using FileIO
using EM
using JLD2
include("hmm.jl")

run_hmm_em_stay_spatial (generic function with 1 method)

In [108]:
function show_results(results; mu=true, sigma=true)
    io = IOBuffer()
    
    print(io, "<pre>")
    print(io, "$(results.varnames)\n")
    print(io, "</pre>")

    if mu
        print(io, "β:<br/>")
        print(io, "<pre>")
        show(io, MIME"text/plain"(), round.(results.betas; digits=2))
        print(io, "</pre>")
    end
    
    if sigma
        print(io, "σ²:<br/>")
        print(io, "<pre>")
        show(io, MIME"text/plain"(), round.(results.sigma; digits=2))
        print(io, "</pre>")
    end
    
    my_string = String(take!(io))
    my_string = replace(my_string, r"[0-9]+×[0-9]+ Matrix{Float64}:\n" => "")
    my_string = replace(my_string, r"(-[0-9\.]+)" => s"<span style='color: red'>\1</span>")
    my_string = replace(my_string, "\n" => "<br/>")
    
    my_string
end

show_results (generic function with 1 method)

In [109]:
function find_Q_vals_turn(animal, results)
    data = load_animal(animal)
    ndays = maximum(data.daynum)
    liks = zeros(ndays)
    Q = zeros(size(data, 1), 6)
    Qstem = zeros(size(data, 1), 3)
    qi = 1
    for i in 1:ndays
        (liks[i], Q_day, Qstem_day) = hmm_Q(view(data, data.daynum .== i, :),
            results.x[i,1],
            results.x[i,2],
            results.x[i,3],
            results.x[i,4],
            results.x[i,5],
            [0.0, 0.0, 0.0],
            0.1 + 0.1 * erf(results.x[i,6] / sqrt(2)),
            get_contingencies()
            )
        Q[qi:qi+size(Q_day, 1)-1, :] .= Q_day
        Qstem[qi:qi+size(Q_day, 1)-1, :] .= Qstem_day
        qi += size(Q_day, 1)
    end
    data[!, "Q1"] = Q[:,1]
    data[!, "Q2"] = Q[:,2]
    data[!, "Q3"] = Q[:,3]
    data[!, "Q4"] = Q[:,4]
    data[!, "Q5"] = Q[:,5]
    data[!, "Q6"] = Q[:,6]
    data[!, "QStem1"] = Qstem[:,1]
    data[!, "QStem2"] = Qstem[:,2]
    data[!, "QStem3"] = Qstem[:,3]
    data
end

find_Q_vals_turn (generic function with 1 method)

In [110]:
hmm_em_stay_turn_peanut = run_hmm_em_stay_turn("peanut")
Q = find_Q_vals_turn("peanut", hmm_em_stay_turn_peanut)
CSV.write("results/Q_vals_turn_peanut.csv", Q)
save("results/hmm_em_stay_turn_peanut.jld2", "hmm_em_stay_turn_peanut", hmm_em_stay_turn_peanut)
HTML(show_results(hmm_em_stay_turn_peanut))

HTML{String}("<pre>[\"βgo\", \"βstay\", \"βleaf\", \"stay_bias\", \"turn_bias\", \"volatility\"]<br/></pre>β:<br/><pre> 13.25  11.44  5.29  3.78  <span style='color: red'>-0.77</span>  1.83</pre>σ²:<br/><pre> 22.79  15.16   1.89   5.26  <span style='color: red'>-2.57</span>  1.9<br/> 15.16  11.87  <span style='color: red'>-0.75</span>   2.96  <span style='color: red'>-1.89</span>  1.48<br/>  1.89  <span style='color: red'>-0.75</span>  15.64   1.15   1.77  2.76<br/>  5.26   2.96   1.15   1.42  <span style='color: red'>-0.48</span>  0.43<br/> <span style='color: red'>-2.57</span>  <span style='color: red'>-1.89</span>   1.77  <span style='color: red'>-0.48</span>   0.61  0.19<br/>  1.9    1.48   2.76   0.43   0.19  0.84</pre>")

In [111]:
hmm_em_stay_turn_senor = run_hmm_em_stay_turn("senor")
Q = find_Q_vals_turn("senor", hmm_em_stay_turn_senor)
CSV.write("results/Q_vals_turn_senor.csv", Q)
save("results/hmm_em_stay_turn_senor.jld2", "hmm_em_stay_turn_senor", hmm_em_stay_turn_senor)
HTML(show_results(hmm_em_stay_turn_senor))

HTML{String}("<pre>[\"βgo\", \"βstay\", \"βleaf\", \"stay_bias\", \"turn_bias\", \"volatility\"]<br/></pre>β:<br/><pre> 12.97  9.63  3.63  5.12  <span style='color: red'>-0.33</span>  1.28</pre>σ²:<br/><pre> 28.77   2.45  15.41   9.79  <span style='color: red'>-4.96</span>   3.0<br/>  2.45   9.54  <span style='color: red'>-5.44</span>  <span style='color: red'>-2.71</span>  <span style='color: red'>-0.28</span>   2.61<br/> 15.41  <span style='color: red'>-5.44</span>  15.83   7.8   <span style='color: red'>-3.59</span>  <span style='color: red'>-0.74</span><br/>  9.79  <span style='color: red'>-2.71</span>   7.8    4.72  <span style='color: red'>-1.75</span>   0.01<br/> <span style='color: red'>-4.96</span>  <span style='color: red'>-0.28</span>  <span style='color: red'>-3.59</span>  <span style='color: red'>-1.75</span>   1.16  <span style='color: red'>-0.21</span><br/>  3.0    2.61  <span style='color: red'>-0.74</span>   0.01  <span style='color: red'>-0.21</span>   1.43</pre>")

In [ ]:
hmm_em_stay_turn_chimi = run_hmm_em_stay_turn("chimi")
Q = find_Q_vals_turn("chimi", hmm_em_stay_turn_chimi)
CSV.write("results/Q_vals_turn_chimi.csv", Q)
save("results/hmm_em_stay_turn_chimi.jld2", "hmm_em_stay_turn_chimi", hmm_em_stay_turn_chimi)
HTML(show_results(hmm_em_stay_turn_chimi))

In [ ]:
hmm_em_stay_turn_j16 = run_hmm_em_stay_turn("j16")
Q = find_Q_vals_turn("j16", hmm_em_stay_turn_j16)
CSV.write("results/Q_vals_turn_j16.csv", Q)
save("results/hmm_em_stay_turn_j16.jld2", "hmm_em_stay_turn_j16", hmm_em_stay_turn_j16)
HTML(show_results(hmm_em_stay_turn_j16))

In [ ]:
hmm_em_stay_turn_wilbur = run_hmm_em_stay_turn("wilbur")
Q = find_Q_vals_turn("wilbur", hmm_em_stay_turn_wilbur)
CSV.write("results/Q_vals_turn_wilbur.csv", Q)
save("results/hmm_em_stay_turn_wilbur.jld2", "hmm_em_stay_turn_wilbur", hmm_em_stay_turn_wilbur)
HTML(show_results(hmm_em_stay_turn_wilbur))

In [ ]:
p = plot()
for i in 1:6
    plot!(hmm_em_stay_turn_peanut.x[:, i], label=hmm_em_stay_turn_peanut.varnames[i], legend=:topleft)
end
title!("peanut (by day, stay + turn biases)")
xlabel!("day")
ylabel!("β")
p

In [ ]:
p = plot()
for i in 1:6
    plot!(hmm_em_stay_turn_wilbur.x[:, i], label=hmm_em_stay_turn_wilbur.varnames[i], legend=:topleft)
end
title!("wilbur (by day, stay + turn biases)")
xlabel!("day")
ylabel!("β")
p

In [ ]:
p = plot()
for i in 1:6
    plot!(hmm_em_stay_turn_j16.x[:, i], label=hmm_em_stay_turn_j16.varnames[i], legend=:topleft)
end
title!("j16 (by day, stay + turn biases)")
xlabel!("day")
ylabel!("β")
p

In [ ]:
p = plot()
for i in 1:6
    plot!(hmm_em_stay_turn_chimi.x[:, i], label=hmm_em_stay_turn_chimi.varnames[i], legend=:topleft)
end
title!("chimi (by day, stay + turn biases)")
xlabel!("day")
ylabel!("β")
p

In [ ]:
p = plot()
for i in 1:6
    plot!(hmm_em_stay_turn_senor.x[:, i], label=hmm_em_stay_turn_senor.varnames[i], legend=:topleft)
end
title!("senor (by day, stay + turn biases)")
xlabel!("day")
ylabel!("β")
p